In [8]:
from pathlib import Path
import sys
import h5py
import numpy as np
import pandas as pd

In [1]:
print("Python:", sys.executable)
print("h5py:", h5py.__version__)

data_dir = Path(
    "/dvs_ro/cfs/cdirs/dunepro/www/data/nd-production/FSDCube/"
    "Reflow_FSDCube_v3/run-ndlar-flow/Reflow_FSDCube_v3.flow/"
    "FLOW/Feb2026/cold/scan/lt_prc8"
)

input_file = data_dir / (
    "Reflow_FSDCube_v3.flow.binary-cosmics-"
    "2026_02_27_17_17_31_PST.FLOW.hdf5"
)

print("Input:", input_file)
print("File exists:", input_file.is_file())

Python: /global/common/software/nersc/pe/conda-envs/26.8.1/python-3.13/nersc-python/bin/python
h5py: 3.16.0
Input: /dvs_ro/cfs/cdirs/dunepro/www/data/nd-production/FSDCube/Reflow_FSDCube_v3/run-ndlar-flow/Reflow_FSDCube_v3.flow/FLOW/Feb2026/cold/scan/lt_prc8/Reflow_FSDCube_v3.flow.binary-cosmics-2026_02_27_17_17_31_PST.FLOW.hdf5
File exists: True


In [4]:
with h5py.File(input_file, "r", locking=False) as f:
    print("File attributes:", dict(f.attrs))

    def inspect(name, obj):
        if isinstance(obj, h5py.Dataset) and name.startswith("charge/"):
            print(f"\n{name}")
            print("  shape:", obj.shape)
            print("  dtype:", obj.dtype)

        elif isinstance(obj, h5py.Group) and name in (
            "geometry_info",
            "run_info",
            "charge/calib_prompt_hits",
        ):
            print(f"\n{name} attributes:", dict(obj.attrs))

    f.visititems(inspect)

File attributes: {}

charge/calib_prompt_hits attributes: {'adc_droop_calibration': np.False_, 'class_version': '1.0.0', 'classname': 'CalibHitBuilder', 'configuration_file': '', 'gain_file': '', 'packets_dset': 'charge/packets', 'pedestal_file': '/global/cfs/cdirs/dune/www/data/LBL/v3/FSDCube/Feb2026/cold_processed/pedestal_prc-1280_rolling-2026_02_26_16_18_05_PST.flow.json', 'source_dset': 'charge/raw_events', 't0_dset': 'combined/t0'}

charge/calib_prompt_hits/data
  shape: (1254632,)
  dtype: [('id', '<u4'), ('x', '<f8'), ('y', '<f8'), ('z', '<f8'), ('t_drift', '<f8'), ('ts_pps', '<u8'), ('io_group', '<u8'), ('io_channel', '<u8'), ('chip_id', '<u8'), ('channel_id', '<u8'), ('Q_raw', '<f8'), ('Q', '<f8'), ('E', '<f8'), ('is_disabled', '?')]

charge/calib_prompt_hits/ref/charge/events/ref_region
  shape: (1254632,)
  dtype: [('start', '<i4'), ('stop', '<i4')]

charge/calib_prompt_hits/ref/charge/packets/ref
  shape: (1254632, 2)
  dtype: uint32

charge/calib_prompt_hits/ref/charge/pa

In [9]:
def h5_tree(val, pre='',skip_ref=False):
    items = len(val)
    for key, val in val.items():
        items -= 1
        if items == 0:
            # the last item
            if type(val) == h5py._hl.group.Group:
                print(pre + '└── ' + key)
                if key=="ref" and skip_ref:
                    continue
                h5_tree(val, pre+'    ')
            else:
                print(pre + '└── ' + key + ' (%d)' % len(val))
        else:
            if type(val) == h5py._hl.group.Group:
                print(pre + '├── ' + key)
                h5_tree(val, pre+'│   ')
            else:
                print(pre + '├── ' + key + ' (%d)' % len(val))

def read_event_hits(f, event_row):
    base = "charge/events/ref/charge/calib_prompt_hits"
    region = f[f"{base}/ref_region"][event_row]
    links = f[f"{base}/ref"][
        int(region["start"]):int(region["stop"])
    ]

    # Source and target are dataset row indices.
    rows = links[links[:, 0] == event_row, 1].astype(np.int64)

    if len(np.unique(rows)) != len(rows):
        raise ValueError(f"Duplicate hit references in event row {event_row}")

    rows = np.sort(rows)  # h5py requires increasing fancy indices
    hits = f["charge/calib_prompt_hits/data"][rows]

    df = pd.DataFrame.from_records(hits)
    df.insert(0, "hit_row", rows)
    df.insert(0, "event_row", event_row)
    return df


In [7]:
with h5py.File(input_file, "r", locking=False) as f:
    h5_tree(f)

├── charge
│   ├── calib_prompt_hits
│   │   ├── data (1254632)
│   │   └── ref
│   │       └── charge
│   │           ├── events
│   │           │   └── ref_region (1254632)
│   │           ├── packets
│   │           │   ├── ref (1254632)
│   │           │   └── ref_region (1254632)
│   │           └── raw_events
│   │               └── ref_region (1254632)
│   ├── events
│   │   ├── data (13888)
│   │   └── ref
│   │       ├── charge
│   │       │   ├── calib_prompt_hits
│   │       │   │   ├── ref (1254632)
│   │       │   │   └── ref_region (13888)
│   │       │   ├── ext_trigs
│   │       │   │   ├── ref (475)
│   │       │   │   └── ref_region (13888)
│   │       │   ├── raw_events
│   │       │   │   └── ref_region (13888)
│   │       │   └── raw_hits
│   │       │       ├── ref (1254632)
│   │       │       └── ref_region (13888)
│   │       └── combined
│   │           └── t0
│   │               ├── ref (13888)
│   │               └── ref_region (13888)
│   ├── ext_trigs
│   

In [10]:
with h5py.File(input_file, "r", locking=False) as f:
    events = f["charge/events/data"][:]
    candidates = np.flatnonzero(
        (events["nhit"] >= 100) & (events["nhit"] <= 500)
    )
    event_row = int(candidates[0]) if len(candidates) else 0

    hits = read_event_hits(f, event_row)
    tick_us = float(f["run_info"].attrs["crs_ticks"])

    print("Event:", events[event_row])
    print("Referenced hits:", len(hits))
    print("Matches nhit:", len(hits) == int(events[event_row]["nhit"]))
    print("Disabled hits:", hits["is_disabled"].sum())
    print("Q equals Q_raw:",
          np.allclose(hits["Q"], hits["Q_raw"], equal_nan=True))

hits["t_drift_us"] = hits["t_drift"] * tick_us
display(hits.head(20))
display(hits[["x", "y", "z", "Q", "t_drift_us"]].describe())

Event: (7, 130, 5300.0, 9256270.0, 9257192.0, 0, 1772241452, 925627)
Referenced hits: 130
Matches nhit: True
Disabled hits: 0
Q equals Q_raw: True


,event_row,hit_row,id,x,y,z,t_drift,ts_pps,io_group,io_channel,chip_id,channel_id,Q_raw,Q,E,is_disabled,t_drift_us
0,7,176,176,48.067501,119.225998,13.578000,0.0,9256270,1,2,130,15,15.873287,15.873287,0.594147,False,0.0
1,7,177,177,48.001244,119.598000,13.950000,6.0,9256276,1,2,130,9,19.492161,19.492161,0.730002,False,0.6
2,7,178,178,47.990202,119.970001,14.322000,7.0,9256277,1,2,130,6,15.050244,15.050244,0.563699,False,0.7
3,7,179,179,47.946030,119.225998,13.206000,11.0,9256281,1,2,130,19,13.534512,13.534512,0.507112,False,1.1
4,7,180,180,47.923945,119.598000,13.578000,13.0,9256283,1,2,130,13,6.263476,6.263476,0.234723,False,1.3
5,7,181,181,47.835603,119.970001,13.950000,21.0,9256291,1,2,130,8,8.381568,8.381568,0.314327,False,2.1
6,7,182,182,47.725175,120.342003,14.694000,31.0,9256301,1,2,130,0,15.433710,15.433710,0.579324,False,3.1
7,7,183,183,47.592661,121.085999,15.066000,43.0,9256313,1,2,120,34,8.791019,8.791019,0.330342,False,4.3
8,7,184,184,47.570576,120.713997,14.694000,45.0,9256315,1,2,130,63,6.610973,6.610973,0.248467,False,4.5
9,7,185,185,47.515362,120.342003,14.322000,50.0,9256320,1,2,130,1,6.877441,6.877441,0.258600,False,5.0


,x,y,z,Q,t_drift_us
count,130.000000,130.000000,130.000000,130.000000,130.000000
mean,43.849074,134.332061,27.253292,10.782316,38.200769
std,2.524676,9.048364,8.209495,4.573435,22.862686
min,37.886056,119.225998,13.206000,2.484977,0.000000
25%,41.670970,125.921997,19.623001,7.322801,17.000000
50%,43.760816,134.292000,27.155999,10.453038,39.000000
75%,46.190228,141.917999,34.316999,13.933598,57.925000
max,48.067501,148.613998,40.734001,24.808550,92.200000


In [11]:
assert (hits["io_channel"] >= 1).all()

hits["io_channel_block"] = (
    (hits["io_channel"].astype("int64") - 1) // 4
)

pixel_key = [
    "event_row", "io_group", "io_channel_block",
    "chip_id", "channel_id"
]

check = hits.groupby(pixel_key).agg(
    n_hits=("id", "size"),
    n_io_channels=("io_channel", "nunique"),
    y_min=("y", "min"),
    y_max=("y", "max"),
    z_min=("z", "min"),
    z_max=("z", "max"),
)
check["y_span"] = check["y_max"] - check["y_min"]
check["z_span"] = check["z_max"] - check["z_min"]

display(check[check["n_hits"] >= 2])

n_hits  n_io_channels  \
event_row io_group io_channel_block chip_id channel_id                          
7         1        0                41      2                2              1   
                                    42      49               2              1   
                                    53      30               2              1   
                                    64      28               2              1   
                                            50               2              1   
                                            55               2              1   
                                            58               2              1   
                                    97      60               2              1   
                                    119     1                2              1   
                                            8                2              1   

                                                             y_min  \
event_row io_group io_channel_block chip_id channel_id               
7         1        0                41      2           146.753998   
                                    42      49          145.638000   
                                    53      30          141.173996   
                                    64      28          137.453995   
                                            50          138.942001   
                                            55          139.313995   
                                            58          139.686005   
                                    97      60          130.386002   
                                    119     1           123.318001   
                                            8           122.945999   

                                                             y_max      z_min  \
event_row io_group io_channel_block chip_id channel_id                          
7         1        0                41      2           146.753998  38.501999   
                                    42      49          145.638000  37.386002   
                                    53      30          141.173996  33.293999   
                                    64      28          137.453995  29.945999   
                                            50          138.942001  31.434000   
                                            55          139.313995  31.806000   
                                            58          139.686005  32.178001   
                                    97      60          130.386002  23.622000   
                                    119     1           123.318001  17.298000   
                                            8           122.945999  16.926001   

                                                            z_max  y_span  \
event_row io_group io_channel_block chip_id channel_id                      
7         1        0                41      2           38.501999     0.0   
                                    42      49          37.386002     0.0   
                                    53      30          33.293999     0.0   
                                    64      28          29.945999     0.0   
                                            50          31.434000     0.0   
                                            55          31.806000     0.0   
                                            58          32.178001     0.0   
                                    97      60          23.622000     0.0   
                                    119     1           17.298000     0.0   
                                            8           16.926001     0.0   

                                                        z_span  
event_row io_group io_channel_block chip_id channel_id          
7         1        0                41      2              0.0  
                                    42      49             0.0  
                                    53      30             0.0  
                   